# Paso 3 — Entrenamiento final

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import random
from torch_geometric.data import Data 
from torch_geometric.transforms import RandomLinkSplit
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score


In [ ]:
BASE_OUTPUT_DIR = './output' 
DATA_FILENAME = 'processed_graph_data.pt'
METADATA_FILENAME = 'metadata.pt'
RESULTS_FILENAME = 'optuna_study_results.csv' 

data_path = os.path.join(BASE_OUTPUT_DIR, DATA_FILENAME)
metadata_path = os.path.join(BASE_OUTPUT_DIR, METADATA_FILENAME)
results_path = os.path.join(BASE_OUTPUT_DIR, RESULTS_FILENAME)

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

# --- Carga de Datos y HPs ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
map_loc = 'cpu' if device.type == 'cpu' else None

print(f"Dispositivo de Cómputo: {device}")

try:
    data = torch.load(data_path, map_location=map_loc, weights_only=False) 
    metadata = torch.load(metadata_path, weights_only=False)
    IN_CHANNELS = data.x.shape[1]
    print(f"✔️ Grafo y Metadata cargados. IN_CHANNELS: {IN_CHANNELS}")
except Exception as e:
    print(f"🚨 Error al cargar datos: {e}")
    data = None
    IN_CHANNELS = None

if data is not None:
    try:
        df_results = pd.read_csv(results_path)
        best_trial_row = df_results.loc[df_results['value_optuna'].idxmax()]
        
        # Extraer HPs 
        HIDDEN_CHANNELS = int(best_trial_row['params_hidden_channels'])
        OUT_CHANNELS = int(best_trial_row['params_out_channels'])
        NUM_HEADS = int(best_trial_row['params_num_heads'])
        PREDICTOR_HIDDEN_CHANNELS = int(best_trial_row['params_predictor_hidden_channels'])
        LEARNING_RATE = float(best_trial_row['params_learning_rate'])
        DROPOUT_RATE = float(best_trial_row['params_dropout_rate'])
        ACTIVATION_FN_NAME = best_trial_row['params_activation_function']
        FINAL_EPOCHS = int(best_trial_row['params_epochs']) 

        print("\n✅ Hiperparámetros Óptimos Cargados:")
        print(f"  Hidden Channels: {HIDDEN_CHANNELS}, Output Channels: {OUT_CHANNELS}, Heads: {NUM_HEADS}")
        print(f"  LR: {LEARNING_RATE}, Dropout: {DROPOUT_RATE}, Épocas: {FINAL_EPOCHS}")

## model architecture

In [ ]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_heads=1, add_self_loops=True, dropout_rate=0.0, activation_fn_name="relu"):
        super(GNNEncoder, self).__init__()
        
        self.conv1 = GATConv(in_channels, hidden_channels, heads=num_heads, 
                              add_self_loops=add_self_loops, edge_dim=1, dropout=dropout_rate)
        
        self.conv2 = GATConv(hidden_channels * num_heads, out_channels, heads=1, 
                              add_self_loops=add_self_loops, edge_dim=1, concat=False, dropout=dropout_rate)
        
        self.dropout_layer = nn.Dropout(dropout_rate) 

        if activation_fn_name == "relu":
            self.activation_fn = F.relu
        elif activation_fn_name == "tanh":
            self.activation_fn = F.tanh
        else:
            raise ValueError(f"Función de activación '{activation_fn_name}' no soportada.")

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = self.activation_fn(x)  
        x = self.dropout_layer(x)  
        
        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        return x

class LinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(LinkPredictor, self).__init__()
        self.lin1 = nn.Linear(in_channels * 2, hidden_channels) 
        self.lin2 = nn.Linear(hidden_channels, out_channels) 

    def forward(self, x_i, x_j):
        x = torch.cat([x_i, x_j], dim=-1) 
        x = self.lin1(x)
        x = F.relu(x) 
        x = self.lin2(x)
        return x

print("✔️ Clases GNNEncoder y LinkPredictor definidas.")

## training evaluation

In [ ]:
def train(model, predictor, data, optimizer, criterion):
    model.train()
    predictor.train()
    optimizer.zero_grad()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Predicción para enlaces positivos de entrenamiento 
    pos_edge_index = data.train_pos_edge_index
    pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])

    # Predicción para enlaces negativos de entrenamiento
    neg_edge_index = data.train_neg_edge_index
    neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])

    # Concatenar predicciones y etiquetas reales
    pred = torch.cat([pos_pred, neg_pred], dim=0)
    target = torch.cat([torch.ones(pos_pred.size(0), device=pred.device), torch.zeros(neg_pred.size(0), device=pred.device)], dim=0)

    # Pérdida y actualización de pesos
    train_loss = criterion(pred.squeeze(), target)
    train_loss.backward()
    optimizer.step()

    # Calcular métricas (en CPU)
    pred_cpu = pred.detach().cpu().numpy().squeeze()
    target_cpu = target.cpu().numpy()
    preds_bin = (pred_cpu >= 0.0).astype(int) # Umbral de 0.0 para logits

    train_auc = roc_auc_score(target_cpu, pred_cpu)
    train_acc = accuracy_score(target_cpu, preds_bin)
    train_precision = precision_score(target_cpu, preds_bin, zero_division=0)
    train_recall = recall_score(target_cpu, preds_bin, zero_division=0)
    train_f1 = f1_score(target_cpu, preds_bin, zero_division=0)

    return train_loss.item(), train_auc, train_acc, train_precision, train_recall, train_f1

@torch.no_grad() 
def test(model, predictor, data):
    model.eval()
    predictor.eval()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Función auxiliar para obtener predicciones y etiquetas
    def get_preds_targets(pos_edge_index, neg_edge_index):
        pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
        neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])
        
        preds_tensor = torch.cat([pos_pred, neg_pred], dim=0).squeeze()
        targets_tensor = torch.cat([torch.ones(pos_pred.size(0), device=preds_tensor.device), torch.zeros(neg_pred.size(0), device=preds_tensor.device)], dim=0)
        
        return preds_tensor.cpu().numpy(), targets_tensor.cpu().numpy()

    # Validación
    val_preds, val_targets = get_preds_targets(data.val_pos_edge_index, data.val_neg_edge_index)
    
    # Prueba
    test_preds, test_targets = get_preds_targets(data.test_pos_edge_index, data.test_neg_edge_index)

    # Loss (Usando tensores en CPU para BCEWithLogitsLoss)
    val_loss = F.binary_cross_entropy_with_logits(torch.tensor(val_preds), torch.tensor(val_targets))
    test_loss = F.binary_cross_entropy_with_logits(torch.tensor(test_preds), torch.tensor(test_targets))

    # Calcular Métricas
    val_auc = roc_auc_score(val_targets, val_preds) 
    test_auc = roc_auc_score(test_targets, test_preds) 

    val_preds_bin = (val_preds >= 0.0).astype(int) 
    test_preds_bin = (test_preds >= 0.0).astype(int)

    val_acc = accuracy_score(val_targets, val_preds_bin)
    test_acc = accuracy_score(test_targets, test_preds_bin)
    val_precision = precision_score(val_targets, val_preds_bin, zero_division=0)
    test_precision = precision_score(test_targets, test_preds_bin, zero_division=0)
    val_recall = recall_score(val_targets, val_preds_bin, zero_division=0)
    test_recall = recall_score(test_targets, test_preds_bin, zero_division=0)
    val_f1 = f1_score(val_targets, val_preds_bin, zero_division=0)
    test_f1 = f1_score(test_targets, test_preds_bin, zero_division=0)

    return val_loss, test_loss, val_auc, test_auc, val_acc, test_acc, val_precision, test_precision, val_recall, test_recall, val_f1, test_f1

print("✔️ Funciones de entrenamiento y evaluación definidas.")

## División de enlaces

In [ ]:
if data is not None:
    set_seed(42) # Usar la misma semilla es CRUCIAL para replicar el split

    print("\nDividiendo enlaces finales para entrenamiento/validación/prueba...")
    
    # RandomLinkSplit debe ser idéntico al usado en 02_tuning
    transform = RandomLinkSplit(
        num_val=0.1,
        num_test=0.1,
        is_undirected=True,
        add_negative_train_samples=True,
        split_labels=True
    )
    
    train_data, val_data, test_data = transform(data) 

    # Crear el objeto data_final que el GNN usará para propagación (solo con train edges)
    data_final = Data(x=train_data.x, edge_index=train_data.edge_index, edge_attr=train_data.edge_attr)
    
    # Asignar los índices de enlaces para la función de predicción
    data_final.train_pos_edge_index = train_data.pos_edge_label_index
    data_final.train_neg_edge_index = train_data.neg_edge_label_index
    data_final.val_pos_edge_index   = val_data.pos_edge_label_index
    data_final.val_neg_edge_index   = val_data.neg_edge_label_index
    data_final.test_pos_edge_index  = test_data.pos_edge_label_index
    data_final.test_neg_edge_index  = test_data.neg_edge_label_index

    # Mover todo al dispositivo de cómputo
    data_final = data_final.to(device)
    
    print(f"  Total de Nodos: {data_final.x.shape[0]}")
    print(f"  Total de Enlaces Positivos de Prueba: {data_final.test_pos_edge_index.shape[1]}")
    print("✔️ División de datos final completada.")

In [ ]:
if 'data_final' in locals() and data_final is not None:
    set_seed(42)
    start_time = time.time()
    
    print("\n--- Inicializando Modelo Final con HPs Óptimos ---")

    # 1. Inicialización del Modelo con HPs cargados
    model = GNNEncoder(IN_CHANNELS, HIDDEN_CHANNELS, OUT_CHANNELS, 
                       num_heads=NUM_HEADS, dropout_rate=DROPOUT_RATE, 
                       activation_fn_name=ACTIVATION_FN_NAME).to(device)
    predictor = LinkPredictor(OUT_CHANNELS, PREDICTOR_HIDDEN_CHANNELS, 1).to(device)

    optimizer = torch.optim.Adam(list(model.parameters()) + list(predictor.parameters()), lr=LEARNING_RATE)
    criterion = torch.nn.BCEWithLogitsLoss()

    # Variables para guardar el mejor modelo
    best_val_auc = 0.0
    best_model_state = None
    best_predictor_state = None
    best_epoch = 0

    print(f"\n--- Iniciando Entrenamiento Final por {FINAL_EPOCHS} Épocas ---")
    
    # 2. Ciclo de Entrenamiento
    for epoch in range(1, FINAL_EPOCHS + 1):
        train_loss, train_auc, _, _, _, _ = train(model, predictor, data_final, optimizer, criterion)
        val_loss, test_loss, val_auc, test_auc, _, _, _, _, _, _, _, _ = test(model, predictor, data_final)
        
        # Guardar el mejor modelo basado en el AUC de validación
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_model_state = model.state_dict()
            best_predictor_state = predictor.state_dict()
            print(f"   >>> Nuevo mejor modelo guardado en Época {epoch} | Val AUC: {val_auc:.4f}")
        
        if epoch % 20 == 0 or epoch == 1 or epoch == FINAL_EPOCHS:
            print(f'  Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | Test AUC: {test_auc:.4f}')

    end_time = time.time()
    total_time = end_time - start_time
    print(f"\n✔️ Entrenamiento final completado en {total_time:.2f} segundos.")

    # 3. Cargar el Mejor Modelo y Evaluar en Test
    if best_model_state and best_predictor_state:
        print(f"\n--- Evaluación Final del Mejor Modelo (Época {best_epoch}) ---")
        
        model.load_state_dict(best_model_state)
        predictor.load_state_dict(best_predictor_state)
        
        final_metrics = evaluate_final_model(model, predictor, data_final, device)
        
        # Mostrar y guardar resultados
        print("\n--- Reporte Final de Métricas (Conjunto de PRUEBA) ---")
        for metric, value in final_metrics.items():
            print(f"  {metric}: {value:.4f}")

        # Guardar el mejor modelo entrenado
        model_save_path = os.path.join(BASE_OUTPUT_DIR, 'final_gnn_encoder.pt')
        predictor_save_path = os.path.join(BASE_OUTPUT_DIR, 'final_link_predictor.pt')
        torch.save(model.state_dict(), model_save_path)
        torch.save(predictor.state_dict(), predictor_save_path)
        print(f"\n✅ Modelo GNN y Predictor guardados en '{BASE_OUTPUT_DIR}'")

        # Guardar el reporte de métricas en un CSV
        df_metrics = pd.DataFrame([final_metrics])
        metrics_save_path = os.path.join(BASE_OUTPUT_DIR, 'final_metrics_report.csv')
        df_metrics.to_csv(metrics_save_path, index=False)
        print(f"✅ Reporte de métricas finales guardado en '{metrics_save_path}'")
    else:
        print("🚨 Fallo al entrenar: No se pudo guardar el mejor modelo.")
else:
    print("🚨 ERROR: 'data_final' no está disponible. Asegúrate de que la Celda 2 se ejecutó correctamente.")